# M1: Audio-Datei

In [ ]:
import numpy as np
from IPython.display import Audio, display
from matplotlib import pyplot as plt
from plotly import graph_objects as go
from plotly.subplots import make_subplots
from scipy.fft import fft, fftfreq
from scipy.io import wavfile

In [ ]:
# Signal einlesen (16 bit -> [-1, 1]) und abspielen
sampling_rate, audio_signal = wavfile.read("audio01.wav")
audio_signal = audio_signal * np.power(2.0, -15)
print(f"{audio_signal.size=}")
print(f"{sampling_rate=}")
print(f"Dauer: {audio_signal.size / sampling_rate} s")
display(Audio(audio_signal, rate=sampling_rate))

## a) - c) Frequenzen und Gewichtung

Wir transformieren die erste Sekunde ($N = f_s$ Werte). Dann ist die Frequenzauflösung $\Delta f = f_s / N = 1\,\mathrm{Hz}$ und der Index $k$ ist direkt die Frequenz in Hz.

Ein reeller Sinus mit Amplitude $A$ erscheint als zwei Linien der Höhe $A/2$ bei $\pm f$.
Die zweite Hälfte der FFT enthält die negativen Frequenzen (Spiegelbild).
Für das einseitige Spektrum nehmen wir die erste Hälfte und **verdoppeln** die Beträge.

In [ ]:
# Spektrum der ersten Sekunde, normiert auf die Anzahl der Abtastwerte
spectrum = fft(audio_signal[0:sampling_rate]) / sampling_rate
frequency = fftfreq(spectrum.size, 1 / sampling_rate)

# Einseitiges Amplitudenspektrum: positive Hälfte, mal 2 (außer Gleichanteil)
half = spectrum.size // 2
amplitude = 2 * np.abs(spectrum[:half])
amplitude[0] /= 2

peaks = np.flatnonzero(amplitude > 0.01)
print(f"{peaks.size} Frequenzen gefunden:")
for k in peaks:
    share = amplitude[k] / amplitude[peaks].sum()
    phase = np.angle(spectrum[k]) / np.pi
    print(
        f"  {frequency[k]:7.0f} Hz  Amplitude {amplitude[k]:.2f}  Anteil {share:4.0%}  Phase {phase:+.2f} pi"
    )

Phase $-\pi/2$ bedeutet $X[k] = -jA/2$: Alle Anteile sind Sinusschwingungen (vgl. Aufgabe 2.4).

## d) Zeitbereich und Frequenzbereich in einer Figure

Einmal interaktiv mit Plotly (zoomen, Werte per Mouse-over ablesen) und einmal mit matplotlib.

In [ ]:
# Zeitbereich von 0 bis 0.1 s
end_i = int(np.ceil(0.1 * sampling_rate))
time_samples = np.arange(end_i + 1) / sampling_rate

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.15)
fig.add_trace(
    go.Scatter(x=time_samples, y=audio_signal[: end_i + 1], name="Signal"), row=1, col=1
)
fig.add_trace(
    go.Scatter(x=frequency[:half], y=amplitude, name="Spektrum"), row=2, col=1
)
fig.update_xaxes(title="Zeit", ticksuffix=" s", row=1, col=1)
fig.update_yaxes(title="Amplitude", row=1, col=1)
fig.update_xaxes(title="Frequenz", ticksuffix=" Hz", row=2, col=1)
fig.update_yaxes(title="Amplitude (einseitig)", row=2, col=1)
fig.update_layout(height=700, showlegend=False)
fig.show()

In [ ]:
fig, (ax_time, ax_freq) = plt.subplots(2, 1, figsize=(10, 7), layout="constrained")
ax_time.plot(time_samples, audio_signal[: end_i + 1], linewidth=0.5)
ax_time.set(xlabel="Zeit in s", ylabel="Amplitude", title="Zeitbereich")
ax_freq.stem(frequency[peaks], amplitude[peaks], basefmt="k-")
ax_freq.set(
    xlabel="Frequenz in Hz",
    ylabel="Amplitude (einseitig)",
    title="Frequenzbereich",
    xlim=(0, sampling_rate / 2),
)
plt.show()